# Aula 15 · Tarefa condicional e iterativa

## O que é
Imagine o **roteiro de uma viagem** por três cidades:
- "**Se chover**, visitamos o museu. **Senão**, vamos à praia." → uma **decisão**.
- "**Em cada cidade**, fazemos check-in no hotel." → uma **repetição**.
- "**Aconteça o que acontecer**, no fim do dia escrevemos no diário." → um passo que roda **sempre**.

No Lakeflow Jobs:

| Na viagem | No job | Para que serve |
|---|---|---|
| "Se chover..." | Tarefa **If/else condition** | Escolher um caminho com base em um valor |
| "Aconteça o que acontecer..." | Dependência **Run if** | Definir quando uma tarefa roda (ex.: *All done*, mesmo com falha antes) |
| "Em cada cidade..." | Tarefa **For each** | Repetir a mesma tarefa para uma lista de entradas |
| A previsão do tempo | **Task values** | Um "recado" que uma tarefa deixa para as próximas |
| A lista de cidades | **Parâmetros dinâmicos** | Valores preenchidos na hora (`{{input}}`, `{{job.run_id}}`, `{{tasks.x.values.y}}`) |

### Run if: as opções
| Opção | Roda quando... |
|---|---|
| **All succeeded** (padrão) | todas as dependências deram certo |
| **All done** | todas terminaram, com sucesso ou falha |
| **At least one succeeded** | pelo menos uma deu certo |
| **None failed** | nenhuma falhou (pode ter sido pulada) |
| **At least one failed** / **All failed** | para tarefas de tratamento de erro |

## Por que importa
Regras de negócio reais exigem **decisões** ("só publique dados que passaram na validação") e **repetições** ("processe cada região"), **sem duplicar tarefas**.

## Como aplicar: desenhando o DAG antes de implementar
No nosso projeto:

1. **Repetição:** a bronze faz a mesma coisa para 4 tabelas → **For each** sobre `["customers", "accounts", "account_balances", "loans"]`.
2. **Recado:** `04_validar_qualidade` mede a silver e publica o task value `qualidade_ok` (verdadeiro/falso).
3. **Decisão:** um **If/else** lê `qualidade_ok`:
   - **verdadeiro** → publica a `gold`;
   - **falso** → `registrar_alerta_qualidade`.
4. **Sempre:** `registrar_execucao` com **Run if: All done**.

```
preparar_ambiente
       ↓
bronze  [For each: customers | accounts | account_balances | loans]
       ↓
silver
       ↓
validar_qualidade   (publica qualidade_ok)
       ↓
qualidade_ok == true ?   [If/else]
   ├── true  → gold
   └── false → registrar_alerta_qualidade
                    ↓  (Run if: All done)
            registrar_execucao
```

> Desenhe este DAG no papel antes da próxima aula. Construir o job fica muito mais fácil quando o desenho está pronto.

## Checklist da aula
- [ ] Sei explicar If/else, For each e Run if com a analogia da viagem
- [ ] Sei o que é um task value e como ele alimenta uma decisão
- [ ] Desenhei o DAG dinâmico do projeto